# Porównanie architektur: MobileNetV2 vs ResNet18 (Transfer learning, ONNX, INT8) — Alfabet ASL

Notatnik realizuje pełny potok (pipeline) uczenia i optymalizacji dwóch różnych sieci neuronowych w celu klasyfikacji alfabetu języka migowego (ASL).

**Główne kroki:**
1. Przygotowanie zbioru danych (ASL Alphabet, 29 klas) z Kaggle.
2. Automatyczny pipeline (`run_pipeline`): transfer learning (trening głowy) → eksport do formatu ONNX → kwantyzacja statyczna do ONNX INT8.
3. Weryfikacja predykcji, dokładności oraz pomiar czasu inferencji.
4. Generowanie paczek `.zip` z artefaktami dla backendu (FastAPI) oraz końcowej tabeli porównawczej dla obu modeli.

> **środowisko:** Trening realizowany jest na **GPU** dla oszczędności czasu, jednak docelowy benchmark (porównanie prędkości PyTorch vs ONNX FP32 vs ONNX INT8) odbywa się na **CPU**, co symuluje docelowe środowisko wdrożeniowe.
>
> **Wykonywanie:** W celu zachowania czystego stanu pamięci i uniknięcia nadpisywania zmiennych, notatnik najlepiej uruchamiać w całości opcją: `Runtime → Restart session and run all`.

In [1]:
# Instalacja zaleznosci (Colab ma juz torch + torchvision z CUDA)
!pip install -q onnx onnxruntime kaggle pandas

In [2]:
import os, time, random, json, warnings, zipfile
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights
from torchvision.models import resnet18, ResNet18_Weights

import onnxruntime as ort
from onnxruntime.quantization import (
    quantize_static, CalibrationDataReader, QuantType, QuantFormat
)
from onnxruntime.quantization.shape_inference import quant_pre_process

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Urzadzenie treningowe:", device)

Urzadzenie treningowe: cuda


## 1–2. Pobranie zbioru danych z Kaggle

Pobieramy dataset z Kaggle [asl-alphabet](https://www.kaggle.com/datasets/grassknoted/asl-alphabet)

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# !unzip -q -o "/content/asl-alphabet.zip" -d /content/data
!unzip -q -o "/content/drive/MyDrive/Colab Notebooks/datasets/asl-alphabet.zip" -d /content/data

print("Pobrano i rozpakowano.")

Pobrano i rozpakowano.


In [5]:
# Konfiguracja
TRAIN_DIR = "data/asl_alphabet_train/asl_alphabet_train"  # jesli sciezka inna, popraw
print("Przyklad klas:", sorted(os.listdir(TRAIN_DIR))[:8], "...")
print("Liczba klas:", len(os.listdir(TRAIN_DIR)))

N_PER_CLASS = 400   # podzbior na klase - zmniejsz, jesli trening za wolny
IMG_SIZE = 224
BATCH = 64
EPOCHS = 8

Przyklad klas: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H'] ...
Liczba klas: 29


In [6]:
mean = [0.485, 0.456, 0.406]   # statystyki ImageNet - MUSZA byc identyczne w aplikacji!
std  = [0.229, 0.224, 0.225]

# UWAGA: dla jezyka migowego NIE odbijamy obrazu w poziomie - zmienia to znaczenie gestu.
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

train_full = datasets.ImageFolder(TRAIN_DIR, transform=train_tf)
eval_full  = datasets.ImageFolder(TRAIN_DIR, transform=eval_tf)
class_names = train_full.classes
num_classes = len(class_names)

# Podzbior N obrazow na klase + podzial 80/20 na train/val (rozlaczne!)
from collections import defaultdict
by_class = defaultdict(list)
for idx, (_, label) in enumerate(train_full.samples):
    by_class[label].append(idx)

train_idx, val_idx = [], []
for label, idxs in by_class.items():
    chosen = random.sample(idxs, min(N_PER_CLASS, len(idxs)))
    split = int(0.8 * len(chosen))
    train_idx += chosen[:split]
    val_idx   += chosen[split:]

train_ds = Subset(train_full, train_idx)
val_ds   = Subset(eval_full,  val_idx)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2)

# class_names sa wspolne dla obu sieci - zapisujemy raz, trafia do kazdego zipa
with open("class_names.json", "w") as f:
    json.dump(class_names, f, ensure_ascii=False)

print(f"Treningowe: {len(train_ds)}, walidacyjne: {len(val_ds)}, klasy: {num_classes}")

Treningowe: 9280, walidacyjne: 2320, klasy: 29


## Funkcje pomocnicze (wspolne dla obu sieci)

In [7]:
def accuracy(model, loader):
    """Dokladnosc modelu PyTorch (model w trybie eval)."""
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            pred = model(imgs).argmax(1)
            correct += (pred == labels).sum().item()
            total += labels.size(0)
    return correct / total

def onnx_accuracy(session, loader):
    """Dokladnosc modelu ONNX (FP32 lub INT8) na calym zbiorze."""
    correct = total = 0
    for imgs, labels in loader:
        out = session.run(None, {"input": imgs.numpy().astype(np.float32)})[0]
        correct += int((out.argmax(1) == labels.numpy()).sum())
        total += labels.size(0)
    return correct / total

def benchmark(fn, x, n=100, warmup=10):
    """Sredni czas inferencji [ms] z rozgrzewka."""
    for _ in range(warmup):
        fn(x)
    t0 = time.perf_counter()
    for _ in range(n):
        fn(x)
    return (time.perf_counter() - t0) / n * 1000.0

def size_mb(p):
    return round(os.path.getsize(p) / 1e6, 2)

# Reader kalibracyjny: probki z TRENINGU (nie z walidacji!) - bez wycieku danych
class ASLCalibrationReader(CalibrationDataReader):
    def __init__(self, loader, n_batches=4):
        self.data = []
        for i, (imgs, _) in enumerate(loader):
            if i >= n_batches:
                break
            for img in imgs:
                self.data.append({"input": img.unsqueeze(0).numpy().astype(np.float32)})
        self.it = iter(self.data)
    def get_next(self):
        return next(self.it, None)

## Pipeline jednej sieci

`run_pipeline(name, build_fn)` robi cala robote dla jednej architektury i zwraca wiersze do tabeli + sciezke do zipa.

`build_fn` zwraca `(model, head)`, gdzie `head` to modul glowy, ktory trzymamy w trybie `train()` (reszta sieci, w tym BatchNorm, zostaje w `eval()` z zamrozonymi statystykami ImageNet).

In [8]:
def build_mobilenetv2():
    m = mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V1)
    for p in m.features.parameters():            # zamrazamy backbone
        p.requires_grad = False
    in_f = m.classifier[1].in_features
    m.classifier[1] = nn.Linear(in_f, num_classes)   # nowa glowa
    return m, m.classifier                        # glowa = Dropout + Linear

def build_resnet18():
    m = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    for p in m.parameters():                      # zamrazamy backbone
        p.requires_grad = False
    m.fc = nn.Linear(m.fc.in_features, num_classes)  # nowa glowa
    return m, m.fc                                # glowa = Linear


def run_pipeline(name, build_fn, epochs=EPOCHS):
    print(f"\n========== {name} ==========")
    model, head = build_fn()
    model = model.to(device)
    trainable = [p for p in model.parameters() if p.requires_grad]
    print("Trenowalnych tensorow:", len(trainable))

    # --- trening: backbone w eval (BN zamrozony), tylko glowa w train ---
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(trainable, lr=1e-3)
    val_acc = 0.0
    for epoch in range(epochs):
        model.eval()
        head.train()
        running = 0.0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            optimizer.step()
            running += loss.item() * imgs.size(0)
        train_loss = running / len(train_ds)
        val_acc = accuracy(model, val_loader)
        print(f"  Epoka {epoch+1}/{epochs} | loss={train_loss:.4f} | val_acc={val_acc*100:.2f}%")
    acc_torch = val_acc   # dokladnosc PyTorch PO treningu

    # --- zapis wag + sciezki ---
    model_cpu = model.to("cpu").eval()
    pth       = f"{name}_asl.pth"
    onnx_fp32 = f"{name}_asl.onnx"
    onnx_prep = f"{name}_asl_prep.onnx"
    onnx_int8 = f"{name}_asl_int8.onnx"
    torch.save(model_cpu.state_dict(), pth)

    # --- eksport ONNX (wyciszamy tylko DeprecationWarning legacy eksportera) ---
    dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=DeprecationWarning)
        torch.onnx.export(
            model_cpu, dummy, onnx_fp32,
            input_names=["input"], output_names=["logits"],
            dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
            opset_version=13, dynamo=False,
        )

    # --- parytet PyTorch vs ONNX ---
    sess = ort.InferenceSession(onnx_fp32, providers=["CPUExecutionProvider"])
    xchk = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
    with torch.no_grad():
        to = model_cpu(xchk).numpy()
    oo = sess.run(None, {"input": xchk.numpy()})[0]
    print("  Parytet PyTorch/ONNX, ten sam argmax:", int(to.argmax()) == int(oo.argmax()),
          "| maks. roznica logitow:", float(np.abs(to - oo).max()))

    # --- kwantyzacja INT8: NAJPIERW pre-process (usuwa warning), kalibracja z TRENINGU ---
    quant_pre_process(onnx_fp32, onnx_prep, skip_symbolic_shape=False)
    quantize_static(
        onnx_prep, onnx_int8, ASLCalibrationReader(train_loader),
        quant_format=QuantFormat.QDQ, per_channel=True,
        weight_type=QuantType.QInt8, activation_type=QuantType.QInt8,
    )
    sess_int8 = ort.InferenceSession(onnx_int8, providers=["CPUExecutionProvider"])

    # --- pomiary czasu na PRAWDZIWEJ probce (nie szum) ---
    x_np = next(iter(val_loader))[0][0].unsqueeze(0).numpy()
    def torch_infer(x):
        with torch.no_grad():
            return model_cpu(torch.from_numpy(x))
    t_torch = benchmark(torch_infer, x_np)
    t_onnx  = benchmark(lambda x: sess.run(None, {"input": x}), x_np)
    t_int8  = benchmark(lambda x: sess_int8.run(None, {"input": x}), x_np)

    # --- dokladnosci ONNX na calej walidacji ---
    acc_fp32 = onnx_accuracy(sess, val_loader)
    acc_int8 = onnx_accuracy(sess_int8, val_loader)

    # --- spakowanie artefaktow do zip (to ladujesz do FastAPI) ---
    zip_path = f"{name}_asl_artifacts.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(onnx_int8)
        z.write(onnx_fp32)
        z.write("class_names.json")
    print("  Spakowano:", zip_path)

    # --- wiersze do tabeli (przyspieszenie liczone vs PyTorch TEJ sieci) ---
    rows = [
        {"Siec": name, "Wariant": "PyTorch CPU",   "Czas [ms]": round(t_torch,2), "Rozmiar [MB]": size_mb(pth),       "Dokladnosc [%]": round(acc_torch*100,2)},
        {"Siec": name, "Wariant": "ONNX FP32 CPU", "Czas [ms]": round(t_onnx,2),  "Rozmiar [MB]": size_mb(onnx_fp32), "Dokladnosc [%]": round(acc_fp32*100,2)},
        {"Siec": name, "Wariant": "ONNX INT8 CPU", "Czas [ms]": round(t_int8,2),  "Rozmiar [MB]": size_mb(onnx_int8), "Dokladnosc [%]": round(acc_int8*100,2)},
    ]
    for r in rows:
        r["Przysp. vs PyTorch"] = round(t_torch / r["Czas [ms]"], 2)
    return rows, zip_path

## Siec 1 — MobileNetV2

In [9]:
rows_mobilenet, zip_mobilenet = run_pipeline("mobilenetv2", build_mobilenetv2)


========== mobilenetv2 ==========
Trenowalnych tensorow: 2
  Epoka 1/8 | loss=2.0251 | val_acc=80.17%
  Epoka 2/8 | loss=0.9976 | val_acc=85.91%
  Epoka 3/8 | loss=0.7105 | val_acc=89.66%
  Epoka 4/8 | loss=0.5687 | val_acc=90.65%
  Epoka 5/8 | loss=0.4808 | val_acc=91.81%
  Epoka 6/8 | loss=0.4322 | val_acc=91.25%
  Epoka 7/8 | loss=0.3874 | val_acc=92.84%
  Epoka 8/8 | loss=0.3463 | val_acc=92.63%
  Parytet PyTorch/ONNX, ten sam argmax: True | maks. roznica logitow: 1.049041748046875e-05
  Spakowano: mobilenetv2_asl_artifacts.zip


## Siec 2 — ResNet18

> Ciekawostka do raportu: MobileNetV2 stoi na konwolucjach *depthwise*, dla ktorych ONNX Runtime na CPU nie ma wydajnych jader INT8 — INT8 bywa wiec mniejszy, ale nie szybszy. ResNet18 ma "zwykle" konwolucje, wiec tam INT8 czesciej realnie przyspiesza. Tabela nizej to pokaze.

In [10]:
rows_resnet, zip_resnet = run_pipeline("resnet18", build_resnet18)


========== resnet18 ==========
Trenowalnych tensorow: 2
  Epoka 1/8 | loss=2.4745 | val_acc=61.51%
  Epoka 2/8 | loss=1.4931 | val_acc=75.39%
  Epoka 3/8 | loss=1.1095 | val_acc=79.87%
  Epoka 4/8 | loss=0.9061 | val_acc=83.15%
  Epoka 5/8 | loss=0.7644 | val_acc=85.56%
  Epoka 6/8 | loss=0.6810 | val_acc=86.29%
  Epoka 7/8 | loss=0.6117 | val_acc=88.71%
  Epoka 8/8 | loss=0.5592 | val_acc=88.41%
  Parytet PyTorch/ONNX, ten sam argmax: True | maks. roznica logitow: 1.9073486328125e-06
  Spakowano: resnet18_asl_artifacts.zip


## Porownanie obu sieci

In [11]:
import pandas as pd
df = pd.DataFrame(rows_mobilenet + rows_resnet)
df = df[["Siec", "Wariant", "Czas [ms]", "Rozmiar [MB]", "Dokladnosc [%]", "Przysp. vs PyTorch"]]
df

,Siec,Wariant,Czas [ms],Rozmiar [MB],Dokladnosc [%],Przysp. vs PyTorch
0,mobilenetv2,PyTorch CPU,25.61,9.29,92.63,1.00
1,mobilenetv2,ONNX FP32 CPU,9.23,9.02,92.63,2.77
2,mobilenetv2,ONNX INT8 CPU,9.81,2.64,86.12,2.61
3,resnet18,PyTorch CPU,57.47,44.84,88.41,1.00
4,resnet18,ONNX FP32 CPU,38.76,44.76,88.41,1.48
5,resnet18,ONNX INT8 CPU,42.78,11.31,86.51,1.34


## Pobranie artefaktow do aplikacji FastAPI

Dwa zipy — po jednym na siec. Kazdy zawiera model INT8, model FP32 i `class_names.json`.

In [12]:
from google.colab import files
for z in [zip_mobilenet, zip_resnet]:
    files.download(z)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>